# 画像のストレス判定：手法比較実験

**目標**：SNS 投稿に添付された画像が「読む人にストレスを与えるか」を自動判定する手法を比較する。

## 比較する手法

| # | 手法 | 概要 | オフライン |
|---|------|------|------------|
| 1 | 色特徴量分析 | 輝度・彩度・色温度などの統計量 | ○ |
| 2 | NSFW 検出 | 性的・暴力的コンテンツの識別 | ○ |
| 3 | CLIP ゼロショット | テキストプロンプトで自由に分類 | ○ |
| 4 | 物体検出 | 武器・炎・血など特定物体の検出 | ○ |
| 5 | 顔感情認識 | 人物の表情から感情を推定 | ○ |

## ストレスを与えうる画像の類型

- 暴力・血液・死を連想させる画像
- 強い嫌悪感を引き起こす画像（虫・腐敗・汚物）
- NSFW（性的・露出）
- 暗い・圧迫感のある色調
- 炎上・事故・災害の画像
- 高コントラスト・点滅（光感受性への影響）
- 比較を誘発する画像（豪邸・高級車・完璧な体型）

## セットアップ

In [ ]:
%pip install -q transformers torch torchvision
%pip install -q Pillow numpy matplotlib requests
%pip install -q opencv-python  # 色特徴量・物体検出

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import requests
import cv2
from PIL import Image
from io import BytesIO
from transformers import pipeline

matplotlib.rcParams['font.family'] = 'Hiragino Sans'

def load_image_from_url(url: str) -> Image.Image:
    """URL から画像を読み込む"""
    resp = requests.get(url, timeout=10)
    return Image.open(BytesIO(resp.content)).convert('RGB')

def load_image_from_path(path: str) -> Image.Image:
    """ローカルパスから画像を読み込む"""
    return Image.open(path).convert('RGB')

print('インポート完了')

## テスト画像の準備

**注意**：実際の有害画像は使用しない。サンプルは Unsplash などのフリー画像を使用する。  
実験では、ローカルに保存した画像を `research/data/images/` に配置して使うことを推奨。

In [ ]:
import os

# サンプル画像の定義
# URL は Unsplash のフリー画像（実験用）
# 実際の研究では research/data/images/ にローカル保存した画像を使うこと

SAMPLE_IMAGES = [
    {
        'name': 'sunny_park',
        'category': 'positive',
        'desc': '晴れた公園の風景（明るい・自然）',
        'url': 'https://images.unsplash.com/photo-1517649763962-0c623066013b?w=400'
    },
    {
        'name': 'dark_storm',
        'category': 'potentially_stressful',
        'desc': '嵐・暗雲の空（暗い・圧迫感）',
        'url': 'https://images.unsplash.com/photo-1504608524841-42584120d693?w=400'
    },
    {
        'name': 'fire',
        'category': 'potentially_stressful',
        'desc': '炎・火災のイメージ',
        'url': 'https://images.unsplash.com/photo-1542751371-adc38448a05e?w=400'
    },
    {
        'name': 'calm_ocean',
        'category': 'neutral',
        'desc': '穏やかな海（中立）',
        'url': 'https://images.unsplash.com/photo-1505118380757-91f5f5632de0?w=400'
    },
]

# 画像を読み込む
images = {}
for sample in SAMPLE_IMAGES:
    try:
        images[sample['name']] = {
            'image': load_image_from_url(sample['url']),
            'category': sample['category'],
            'desc': sample['desc'],
        }
        print(f"✓ {sample['name']}: {sample['desc']}")
    except Exception as e:
        print(f"✗ {sample['name']}: 読み込み失敗 ({e})")

# グリッド表示
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, (name, data) in zip(axes, images.items()):
    ax.imshow(data['image'])
    ax.set_title(f"{name}\n{data['desc'][:15]}", fontsize=9)
    ax.axis('off')
plt.suptitle('テスト画像一覧', fontsize=12)
plt.tight_layout()
plt.show()

---
## 手法 1：色特徴量分析

色の統計量からストレスと関連する視覚的特徴を抽出する。  

**根拠**：
- 暗い色調（低輝度）は不安・抑うつと関連（色彩心理学）
- 低彩度（グレーがかった色）はうつ状態の表現に多い
- 赤・オレンジの高彩度は怒り・危険の警告色
- 高コントラスト・突然の明暗変化は視覚的不快感を生じる

In [ ]:
def extract_color_features(pil_image: Image.Image) -> dict:
    """HSV 色空間での統計量を計算してストレス関連特徴を抽出"""
    img_np = np.array(pil_image)
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    h, s, v = img_hsv[:,:,0], img_hsv[:,:,1], img_hsv[:,:,2]

    # 輝度（V）の統計
    v_mean = float(v.mean()) / 255.0      # 平均輝度（0=暗い, 1=明るい）
    v_std  = float(v.std())  / 255.0      # 輝度の分散（高=コントラスト強）

    # 彩度（S）の統計
    s_mean = float(s.mean()) / 255.0      # 平均彩度（0=グレー, 1=鮮やか）

    # 赤系（H=0-10 or 170-180）の占有率
    red_mask = (h < 10) | (h > 170)
    red_ratio = float(red_mask.sum()) / red_mask.size

    # ストレススコアの計算（0〜1）
    # - 暗い（v_mean が低い）: ストレス ↑
    # - 高コントラスト（v_std が高い）: ストレス ↑（急な明暗変化）
    # - 低彩度（s_mean が低い）かつ暗い: ストレス ↑（うつ的トーン）
    # - 赤が多い（red_ratio が高い）: ストレス ↑（危険・警告）
    darkness_score   = 1.0 - v_mean                          # 暗さ
    contrast_score   = min(1.0, v_std * 3.0)                 # コントラスト
    dull_score       = (1.0 - s_mean) * darkness_score       # 暗くてくすんだ色
    red_score        = min(1.0, red_ratio * 5.0)             # 赤系の存在

    stress_score = (
        darkness_score  * 0.35 +
        contrast_score  * 0.25 +
        dull_score      * 0.25 +
        red_score       * 0.15
    )

    return {
        'v_mean': round(v_mean, 3),
        'v_std':  round(v_std, 3),
        's_mean': round(s_mean, 3),
        'red_ratio': round(red_ratio, 3),
        'stress_score': round(stress_score, 3),
        'flagged': stress_score > 0.45
    }

print('=== 色特徴量分析 ===')
color_results = {}
for name, data in images.items():
    feat = extract_color_features(data['image'])
    color_results[name] = feat
    flag = '🔴' if feat['flagged'] else '⚪'
    print(f"{flag} [{data['category']:22s}] score={feat['stress_score']:.3f} "
          f"dark={feat['v_mean']:.2f} sat={feat['s_mean']:.2f} red={feat['red_ratio']:.3f}  "
          f"{data['desc']}")

In [ ]:
# 色ヒストグラムの可視化
fig, axes = plt.subplots(2, len(images), figsize=(16, 6))

for col, (name, data) in enumerate(images.items()):
    img_np = np.array(data['image'])
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    axes[0][col].imshow(data['image'])
    axes[0][col].set_title(name, fontsize=9)
    axes[0][col].axis('off')

    # 輝度（V）のヒストグラム
    axes[1][col].hist(img_hsv[:,:,2].ravel(), bins=50, color='gray', alpha=0.7, density=True)
    axes[1][col].set_title(f"輝度分布\nscore={color_results[name]['stress_score']:.2f}", fontsize=8)
    axes[1][col].set_xlim(0, 255)

plt.suptitle('手法1: 色特徴量分析', fontsize=12)
plt.tight_layout()
plt.show()

---
## 手法 2：NSFW 検出

**モデル**：`Falconsai/nsfw_image_detection`（Vision Transformer ベース）  
**ラベル**：`normal` / `nsfw`  
**用途**：性的・暴力的・露出コンテンツの検出。ストレス判定の中でも最も明確なカテゴリ。

In [ ]:
print('NSFW モデルを読み込み中...')
nsfw_pipeline = pipeline(
    'image-classification',
    model='Falconsai/nsfw_image_detection'
)
print('読み込み完了')

def nsfw_stress(pil_image: Image.Image) -> dict:
    results = nsfw_pipeline(pil_image)
    score_map = {r['label']: r['score'] for r in results}
    nsfw_score = score_map.get('nsfw', 0.0)
    return {
        'nsfw_score': round(nsfw_score, 3),
        'normal_score': round(score_map.get('normal', 0.0), 3),
        'flagged': nsfw_score > 0.5
    }

print('\n=== NSFW 検出 ===')
nsfw_results = {}
for name, data in images.items():
    res = nsfw_stress(data['image'])
    nsfw_results[name] = res
    flag = '🔴' if res['flagged'] else '⚪'
    print(f"{flag} [{data['category']:22s}] nsfw={res['nsfw_score']:.3f}  {data['desc']}")

---
## 手法 3：CLIP ゼロショット分類

**モデル**：`openai/clip-vit-base-patch32`  
**特徴**：テキストプロンプトと画像の類似度を計算。カスタムラベルで自由に分類できる。  
テキストのゼロショット分類と同様に、**仮説文**として与えることで精度が上がる。

In [ ]:
print('CLIP モデルを読み込み中...')
clip_pipeline = pipeline(
    'zero-shot-image-classification',
    model='openai/clip-vit-base-patch32'
)
print('読み込み完了')

# ストレス判定用のラベル
# 英語で指定するほうが CLIP は精度が出やすい（英語で学習されたため）
CLIP_STRESS_LABELS = [
    'a disturbing or violent image',
    'a dark and depressing scene',
    'a stressful and overwhelming image',
    'a calm and peaceful scene',
    'a happy and positive image',
]

# ストレスラベルに対応するインデックス
STRESS_LABEL_INDICES = {0, 1, 2}  # disturbing, dark, stressful

def clip_stress(pil_image: Image.Image) -> dict:
    results = clip_pipeline(pil_image, candidate_labels=CLIP_STRESS_LABELS)
    score_map = {r['label']: r['score'] for r in results}
    top_label = results[0]['label']
    top_score = results[0]['score']
    # ストレス関連ラベルがトップかつスコアが高い場合にフラグ
    stress_labels = CLIP_STRESS_LABELS[:3]
    stress_score = sum(score_map.get(l, 0) for l in stress_labels)
    return {
        'top_label': top_label,
        'top_score': round(top_score, 3),
        'stress_score': round(stress_score, 3),
        'all_scores': {r['label'][:25]: round(r['score'], 3) for r in results},
        'flagged': stress_score > 0.5
    }

print('\n=== CLIP ゼロショット 検出 ===')
clip_results = {}
for name, data in images.items():
    res = clip_stress(data['image'])
    clip_results[name] = res
    flag = '🔴' if res['flagged'] else '⚪'
    print(f"{flag} [{data['category']:22s}] stress={res['stress_score']:.3f} top='{res['top_label'][:30]}'")
    print(f"     {data['desc']}")

In [ ]:
# カスタムラベル実験
# SNS ストレス研究に特化したラベルで試す

CUSTOM_LABELS_V2 = [
    'an image that would make someone feel anxious or stressed',
    'an image showing danger, violence, or disturbing content',
    'an image that evokes sadness, grief, or depression',
    'an image meant to provoke envy or social comparison',
    'an ordinary or neutral everyday image',
    'a beautiful, uplifting, or joyful image',
]

print('カスタムラベル v2 で CLIP 評価')
for name, data in images.items():
    results = clip_pipeline(data['image'], candidate_labels=CUSTOM_LABELS_V2)
    print(f"\n📷 {name} ({data['desc']})")
    for r in results:
        bar = '█' * int(r['score'] * 30)
        print(f"  {r['score']:.3f} {bar:30s} {r['label'][:45]}")

---
## 手法 4：顔感情認識

**モデル**：`trpakov/vit-face-expression`（ViT ベースの顔表情認識）  
**感情ラベル**：`angry / disgust / fear / happy / neutral / sad / surprise`  
**用途**：人物が写っている投稿において、表情からストレス・怒り・悲しみを検出する。  
**限界**：顔が写っていない画像には適用できない。

In [ ]:
print('顔感情認識モデルを読み込み中...')
face_emotion_pipeline = pipeline(
    'image-classification',
    model='trpakov/vit-face-expression'
)
print('読み込み完了')

# ストレスに関連する感情の重みづけ
FACE_STRESS_WEIGHT = {
    'angry':   1.0,
    'fear':    0.9,
    'sad':     0.8,
    'disgust': 0.7,
    'surprise': 0.3,
    'neutral':  0.0,
    'happy':    0.0,
}

def face_stress(pil_image: Image.Image) -> dict:
    try:
        results = face_emotion_pipeline(pil_image)
        score_map = {r['label'].lower(): r['score'] for r in results}
        stress_score = sum(score_map.get(e, 0) * w for e, w in FACE_STRESS_WEIGHT.items())
        top_emotion = max(score_map, key=score_map.get)
        return {
            'stress_score': round(stress_score, 3),
            'top_emotion': top_emotion,
            'flagged': stress_score > 0.4
        }
    except Exception as e:
        return {'stress_score': 0.0, 'top_emotion': 'unknown', 'flagged': False, 'error': str(e)}

print('\n=== 顔感情認識 ===')
print('（顔が写っていない画像では精度が低い点に注意）')
for name, data in images.items():
    res = face_stress(data['image'])
    flag = '🔴' if res['flagged'] else '⚪'
    print(f"{flag} [{data['category']:22s}] score={res['stress_score']:.3f} top={res['top_emotion']}  {data['desc']}")

---
## 比較・可視化

In [ ]:
import pandas as pd

comparison_data = []
for name, data in images.items():
    comparison_data.append({
        'name': name,
        'category': data['category'],
        'desc': data['desc'][:15],
        '色特徴量': color_results.get(name, {}).get('stress_score', float('nan')),
        'NSFW': nsfw_results.get(name, {}).get('nsfw_score', float('nan')),
        'CLIP': clip_results.get(name, {}).get('stress_score', float('nan')),
    })

df_img = pd.DataFrame(comparison_data).set_index('name')

import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    df_img[['色特徴量', 'NSFW', 'CLIP']],
    annot=True, fmt='.3f', cmap='RdYlGn_r',
    vmin=0, vmax=1, ax=ax, linewidths=0.5
)
ax.set_title('画像ストレス判定：手法比較\n（高いほどストレスありと判定）', fontsize=12)
plt.tight_layout()
plt.show()

print(df_img[['category', '色特徴量', 'NSFW', 'CLIP']])

---
## 考察・まとめ

### 各手法の適用範囲

| 手法 | 得意なストレス類型 | 限界 |
|------|-------------------|------|
| 色特徴量 | 暗い・圧迫感・くすんだ色調 | 明るい画像でも有害なことがある |
| NSFW 検出 | 性的・露出コンテンツ | ニュアンスのある不快感は検出できない |
| CLIP ゼロショット | 文脈的なストレス（柔軟） | ラベル設計に依存。精度にばらつき |
| 顔感情認識 | 怒り・悲しみ・恐怖の表情 | 顔が写っていない画像には使えない |

### 推奨アーキテクチャ

```
画像入力
  ↓
Layer 1: 色特徴量（高速・常時オン）
  ↓
Layer 2a: NSFW 検出（NSFW 閾値の超過で即フィルタ）
Layer 2b: CLIP ゼロショット（文脈的ストレスの検出）
  ↓ 顔が含まれる場合のみ
Layer 2c: 顔感情認識（表情からのストレス検出）
```

### 今後の実験課題

1. **比較誘発画像の検出**：高級品・理想的な体型・豪華な食事の画像が CLIP でどれほど検出できるか
2. **フリッカー・高コントラスト**：光感受性てんかんへの影響がある画像の検出
3. **テキスト付き画像**：OCR でテキストを抽出し、テキスト判定と組み合わせる
4. **SNS 実データでの評価**：Bluesky API から取得した画像で誤検知率を測定